In [15]:
import math
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.covariance import LedoitWolf
from sklearn.model_selection import TimeSeriesSplit

In [17]:
PRICES_PATH = "prices.csv"
META_PATH = "meta.csv"

In [19]:
N_ASSETS = 25
TICKS_PER_DAY = 30
TRADING_DAYS_PER_YEAR = 252
IMPACT_MULT = 2.5
DT_YEAR = 1.0 / (TRADING_DAYS_PER_YEAR * TICKS_PER_DAY)

TRAIN_YEARS = 4
HOLDOUT_YEARS = 1
TRAIN_TICKS = TRAIN_YEARS * TRADING_DAYS_PER_YEAR * TICKS_PER_DAY
HOLDOUT_TICKS = HOLDOUT_YEARS * TRADING_DAYS_PER_YEAR * TICKS_PER_DAY

ASSET_COLUMNS = tuple(f"A{i:02d}" for i in range(N_ASSETS))


@dataclass(frozen=True)
class PublicMeta:
    sector_id: np.ndarray
    spread_bps: np.ndarray
    borrow_bps_annual: np.ndarray


def load_prices(path: str = PRICES_PATH) -> np.ndarray:
    df = pd.read_csv(path, index_col="tick")
    return df[list(ASSET_COLUMNS)].to_numpy(dtype=float)


def load_meta(path: str = META_PATH) -> PublicMeta:
    df = pd.read_csv(path)
    return PublicMeta(
        sector_id=df["sector_id"].to_numpy(dtype=int),
        spread_bps=df["spread_bps"].to_numpy(dtype=float),
        borrow_bps_annual=df["borrow_bps_annual"].to_numpy(dtype=float),
    )

In [21]:
def project_to_gross_limit(w: np.ndarray) -> np.ndarray:
    w = np.asarray(w, dtype=float).copy()
    gross = float(np.sum(np.abs(w)))
    if not np.isfinite(gross):
        return np.zeros_like(w)
    if gross > 1.0:
        w /= gross
    return w


def fit_covariances(train_ret: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    x = train_ret.values
    cov_sample = np.cov(x, rowvar=False)
    lw = LedoitWolf().fit(x)
    cov_lw = lw.covariance_
    return cov_sample, cov_lw


def risk_parity_weights(cov: np.ndarray, tol: float = 1e-8, max_iter: int = 1000) -> np.ndarray:
    n = cov.shape[0]
    w = np.ones(n, dtype=float) / n
    eps = 1e-12

    for _ in range(max_iter):
        marginal = cov @ w
        rc = w * marginal
        port_var = float(w @ marginal)

        if port_var <= eps:
            return np.ones(n, dtype=float) / n

        target = port_var / n
        denom = np.maximum(rc, eps)
        w_new = w * (target / denom)
        w_new = np.maximum(w_new, eps)
        w_new /= w_new.sum()

        if np.linalg.norm(w_new - w, ord=1) < tol:
            w = w_new
            break
        w = w_new

    return w


def annualized_sharpe(daily_returns: np.ndarray) -> float:
    x = np.asarray(daily_returns, dtype=float)
    mu = float(np.mean(x))
    sd = float(np.std(x, ddof=1))
    if not np.isfinite(sd) or sd < 1e-12:
        return -np.inf if mu <= 0 else np.inf
    return math.sqrt(TRADING_DAYS_PER_YEAR) * mu / sd

In [23]:
class MyStrategy:
    """
    Improved Case 2 strategy:
    - risk parity base
    - multi-horizon momentum
    - sector-relative alpha
    - volatility scaling
    - smoother daily rebalancing
    - weaker expensive shorts
    """

    def __init__(self):
        self.ticks_per_day = TICKS_PER_DAY
        self.sector_id = None
        self.spread_bps = None
        self.borrow_bps_annual = None

        # alpha / portfolio params
        self.beta = 0.7              # strength of alpha overlay
        self.lookback_cov = 63        # covariance estimation window
        self.min_days = 70            # need enough history for 60d alpha

        # smoothing / turnover control
        self.rebalance_rate = 0.25    # move only partway to target each day
        self.prev_weights = None

        # alpha construction
        self.mom_short = 5
        self.mom_mid = 20
        self.mom_long = 60

        self.cost_penalty = 10.0
        self.short_penalty = 6.0
        self.short_scale = 0.60       # shrink all shorts a bit

    def fit(self, train_prices: np.ndarray, meta: PublicMeta, **kwargs) -> None:
        self.ticks_per_day = int(kwargs.get("ticks_per_day", TICKS_PER_DAY))
        self.sector_id = np.asarray(meta.sector_id, dtype=int)
        self.spread_bps = np.asarray(meta.spread_bps, dtype=float)
        self.borrow_bps_annual = np.asarray(meta.borrow_bps_annual, dtype=float)
        self.prev_weights = None

    def _daily_closes(self, price_history) -> np.ndarray:
        prices = np.asarray(price_history, dtype=float)

        n_ticks = prices.shape[0]
        n_days = n_ticks // self.ticks_per_day
        if n_days == 0:
            return np.empty((0, prices.shape[1]), dtype=float)

        close_idx = np.arange(
            self.ticks_per_day - 1,
            n_days * self.ticks_per_day,
            self.ticks_per_day,
        )
        return prices[close_idx]

    def _daily_returns_df(self, price_history) -> pd.DataFrame:
        closes = self._daily_closes(price_history)
        if closes.shape[0] <= 1:
            return pd.DataFrame(np.empty((0, closes.shape[1])), columns=ASSET_COLUMNS)

        ret = closes[1:] / closes[:-1] - 1.0
        return pd.DataFrame(ret, columns=ASSET_COLUMNS)

    def _sector_neutralize(self, alpha: np.ndarray) -> np.ndarray:
        alpha = alpha.copy()
        for s in np.unique(self.sector_id):
            idx = np.where(self.sector_id == s)[0]
            if len(idx) > 0:
                alpha[idx] -= alpha[idx].mean()
        return alpha

    def _zscore(self, x: np.ndarray) -> np.ndarray:
        x = np.asarray(x, dtype=float)
        mu = np.mean(x)
        sd = np.std(x)
        if sd < 1e-12:
            return np.zeros_like(x)
        return (x - mu) / sd

    def _rolling_mean_signal(self, daily_ret: pd.DataFrame, lb: int) -> np.ndarray:
        lb = min(lb, len(daily_ret))
        if lb <= 0:
            return np.zeros(N_ASSETS, dtype=float)
        return daily_ret.iloc[-lb:].mean(axis=0).values

    def _rolling_vol(self, daily_ret: pd.DataFrame, lb: int = 20) -> np.ndarray:
        lb = min(lb, len(daily_ret))
        if lb <= 1:
            return np.ones(N_ASSETS, dtype=float)
        vol = daily_ret.iloc[-lb:].std(axis=0).values
        vol[vol < 1e-6] = 1e-6
        return vol

    def _build_alpha(self, daily_ret: pd.DataFrame) -> np.ndarray:
        if len(daily_ret) < 10:
            return np.zeros(N_ASSETS, dtype=float)

        # multi-horizon momentum
        mom_5 = self._rolling_mean_signal(daily_ret, self.mom_short)
        mom_20 = self._rolling_mean_signal(daily_ret, self.mom_mid)
        mom_60 = self._rolling_mean_signal(daily_ret, self.mom_long)

        # short-term reversal
        rev_5 = -mom_5

        # blend
        raw_alpha = 0.20 * mom_5 + 0.45 * mom_20 + 0.25 * mom_60 + 0.10 * rev_5

        # sector relative
        raw_alpha = self._sector_neutralize(raw_alpha)

        # volatility scaling
        vol = self._rolling_vol(daily_ret, lb=20)
        raw_alpha = raw_alpha / vol

        # cross-sectional normalize
        alpha = self._zscore(raw_alpha)

        # cost-aware shrinkage
        spread_frac = self.spread_bps / 1e4
        borrow_frac = self.borrow_bps_annual / 1e4

        alpha = alpha / (1.0 + self.cost_penalty * spread_frac)

        neg = alpha < 0
        alpha[neg] = alpha[neg] / (1.0 + self.short_penalty * borrow_frac[neg])

        # globally shrink shorts a little
        alpha[neg] *= self.short_scale

        return alpha

    def get_weights(self, price_history, meta: PublicMeta, day: int) -> np.ndarray:
        daily_ret = self._daily_returns_df(price_history)
        n_days = len(daily_ret)

        if n_days < self.min_days:
            w = np.ones(N_ASSETS, dtype=float) / N_ASSETS
            self.prev_weights = w.copy()
            return w

        # covariance / RP base
        cov_lb = min(self.lookback_cov, n_days)
        train_ret = daily_ret.iloc[-cov_lb:]
        _, cov_lw = fit_covariances(train_ret)

        w_rp = risk_parity_weights(cov_lw)

        # alpha overlay
        alpha = self._build_alpha(daily_ret)

        # --- STEP 1: center
        alpha = alpha - np.mean(alpha)
        
        # --- STEP 2: pick strongest signals only
        n = len(alpha)
        k = int(0.3 * n)
        
        sorted_idx = np.argsort(alpha)
        
        long_idx = sorted_idx[-k:]
        short_idx = sorted_idx[:k]
            
        alpha_filtered = np.zeros(n)
        alpha_filtered[long_idx] = alpha[long_idx]
        alpha_filtered[short_idx] = alpha[short_idx]
            
        # --- STEP 3: normalize
        if np.sum(np.abs(alpha_filtered)) < 1e-12:
            target = w_rp
        else:
            alpha_overlay = alpha_filtered / (np.sum(np.abs(alpha_filtered)) + 1e-12)
            target = w_rp + self.beta * alpha_overlay

        # light shrink toward RP so overlay does not dominate
        target = (1.0 - self.beta) * w_rp + self.beta * alpha_overlay
        target = project_to_gross_limit(target) 

        # smoother rebalancing
        if self.prev_weights is None or len(self.prev_weights) != N_ASSETS:
            w = target
        else:
            w = (1.0 - self.rebalance_rate) * self.prev_weights + self.rebalance_rate * target

        w = project_to_gross_limit(w)

        if not np.all(np.isfinite(w)):
            w = np.ones(N_ASSETS, dtype=float) / N_ASSETS

        self.prev_weights = w.copy()
        return w

In [25]:
def _transaction_cost(
    spread: np.ndarray, delta_weights: np.ndarray, impact_mult: float = IMPACT_MULT
) -> tuple[float, float]:
    linear = float(np.sum((spread / 2.0) * np.abs(delta_weights)))
    quadratic = float(np.sum((impact_mult * spread) * (delta_weights ** 2)))
    return linear, quadratic


def _hold_fixed_weights_one_day(
    wealth: float,
    weights: np.ndarray,
    logret: np.ndarray,
    borrow: np.ndarray,
    *,
    day: int,
) -> float:
    t0 = day * TICKS_PER_DAY
    t_begin = t0 + 1 if day == 0 else t0

    for t in range(t_begin, t0 + TICKS_PER_DAY):
        pnl = float(np.sum(weights * (np.exp(logret[t]) - 1.0)))
        borrow_cost = float(np.sum(np.maximum(-weights, 0.0) * borrow) * DT_YEAR)
        wealth *= 1.0 + pnl - borrow_cost

    return wealth


def _history_through_day(
    train_prices: np.ndarray, hold_prices: np.ndarray, day: int
) -> np.ndarray:
    cutoff = (day + 1) * TICKS_PER_DAY
    return np.vstack([train_prices, hold_prices[:cutoff]])


def run_backtest(
    train_prices,
    hold_prices,
    strategy,
    meta: PublicMeta,
) -> dict:
    train_prices = np.asarray(train_prices, dtype=float)
    hold_prices = np.asarray(hold_prices, dtype=float)

    spread = np.asarray(meta.spread_bps, dtype=float) / 1e4
    borrow = np.asarray(meta.borrow_bps_annual, dtype=float) / 1e4

    strategy.fit(train_prices, meta, ticks_per_day=TICKS_PER_DAY)
    weights = project_to_gross_limit(strategy.get_weights(train_prices, meta, day=0))

    wealth = 1.0
    entry_linear, entry_quadratic = _transaction_cost(spread, weights, IMPACT_MULT)
    wealth *= 1.0 - (entry_linear + entry_quadratic)

    logret = np.zeros_like(hold_prices, dtype=float)
    logret[1:, :] = np.log(hold_prices[1:, :] / hold_prices[:-1, :])

    n_days = hold_prices.shape[0] // TICKS_PER_DAY
    daily_returns = np.zeros(n_days)
    daily_costs = np.zeros(n_days + 1)
    daily_costs[0] = entry_linear + entry_quadratic

    weights_history = [weights.copy()]

    for day in range(n_days):
        wealth_start = wealth
        wealth = _hold_fixed_weights_one_day(wealth, weights, logret, borrow, day=day)

        if wealth <= 0 or not np.isfinite(wealth):
            daily_returns[day:] = -1.0
            return {
                "daily_returns": daily_returns,
                "daily_costs": daily_costs[: day + 1],
                "weights_history": np.array(weights_history),
                "blown_up": True,
            }

        history = _history_through_day(train_prices, hold_prices, day)
        target = project_to_gross_limit(strategy.get_weights(history, meta, day=day + 1))

        delta = target - weights
        linear, quadratic = _transaction_cost(spread, delta, IMPACT_MULT)
        trade_cost = linear + quadratic

        wealth *= 1.0 - trade_cost
        daily_costs[day + 1] = trade_cost
        daily_returns[day] = wealth / wealth_start - 1.0

        weights = target
        weights_history.append(weights.copy())

    return {
        "daily_returns": daily_returns,
        "daily_costs": daily_costs,
        "weights_history": np.array(weights_history),
        "blown_up": False,
    }

In [27]:
train_prices = prices[:TRAIN_TICKS]
hold_prices = prices[TRAIN_TICKS : TRAIN_TICKS + HOLDOUT_TICKS]

print("Train ticks:", train_prices.shape)
print("Holdout ticks:", hold_prices.shape)

NameError: name 'prices' is not defined

In [ ]:
strategy = MyStrategy()
result = run_backtest(train_prices, hold_prices, strategy, meta)

daily_returns = result["daily_returns"]
daily_costs = result["daily_costs"]
weights_history = result["weights_history"]

print("Blew up:", result["blown_up"])
print("Sharpe:", annualized_sharpe(daily_returns))
print("Total return:", np.prod(1.0 + daily_returns) - 1.0)
print("Total txn costs:", np.sum(daily_costs))

In [ ]:
equity = np.cumprod(1.0 + daily_returns)

plt.figure(figsize=(10, 5))
plt.plot(equity)
plt.title("Equity Curve")
plt.xlabel("Day")
plt.ylabel("Portfolio Value")
plt.grid(True)
plt.show()

In [ ]:
cummax = np.maximum.accumulate(equity)
drawdown = equity / cummax - 1.0

plt.figure(figsize=(10, 4))
plt.plot(drawdown)
plt.title("Drawdown")
plt.xlabel("Day")
plt.ylabel("Drawdown")
plt.grid(True)
plt.show()

print("Max drawdown:", drawdown.min())

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(daily_costs)
plt.title("Daily Transaction Costs")
plt.xlabel("Day")
plt.ylabel("Cost")
plt.grid(True)
plt.show()

print("Average daily cost:", np.mean(daily_costs))
print("Total cost:", np.sum(daily_costs))

In [ ]:
final_weights = weights_history[-1]
weights_df = pd.DataFrame({
    "asset": ASSET_COLUMNS,
    "sector": meta.sector_id,
    "spread_bps": meta.spread_bps,
    "borrow_bps_annual": meta.borrow_bps_annual,
    "weight": final_weights
}).sort_values("weight", ascending=False)

weights_df

In [ ]:
plt.figure(figsize=(12, 5))
plt.bar(weights_df["asset"], weights_df["weight"])
plt.xticks(rotation=90)
plt.title("Final Portfolio Weights")
plt.grid(True, axis="y")
plt.show()

In [ ]:
def run_cv(prices: np.ndarray, meta: PublicMeta, n_splits: int = 3) -> pd.DataFrame:
    years_total = prices.shape[0] // (TRADING_DAYS_PER_YEAR * TICKS_PER_DAY)
    ticks_per_year = TRADING_DAYS_PER_YEAR * TICKS_PER_DAY

    rows = []

    # expanding window: test on 1 year
    for k in range(2, years_total):
        train_end = k * ticks_per_year
        test_end = (k + 1) * ticks_per_year

        if test_end > prices.shape[0]:
            break

        train_prices = prices[:train_end]
        hold_prices = prices[train_end:test_end]

        strategy = MyStrategy()
        result = run_backtest(train_prices, hold_prices, strategy, meta)

        dr = result["daily_returns"]
        eq = np.cumprod(1.0 + dr)
        dd = eq / np.maximum.accumulate(eq) - 1.0

        rows.append({
            "test_year": k,
            "sharpe": annualized_sharpe(dr),
            "total_return": np.prod(1.0 + dr) - 1.0,
            "max_drawdown": dd.min(),
            "total_cost": np.sum(result["daily_costs"]),
            "blew_up": result["blown_up"],
        })

    return pd.DataFrame(rows)

In [ ]:
cv_results = run_cv(prices, meta)
cv_results

In [ ]:
print("Mean Sharpe:", cv_results["sharpe"].mean())
print("Min Sharpe:", cv_results["sharpe"].min())
print("Max Sharpe:", cv_results["sharpe"].max())
print("Mean total return:", cv_results["total_return"].mean())
print("Mean cost:", cv_results["total_cost"].mean())

In [ ]:
def evaluate_param_grid_v3(prices, meta, betas, ks, thresholds):
    rows = []
    ticks_per_year = TRADING_DAYS_PER_YEAR * TICKS_PER_DAY

    train_prices = prices[:4 * ticks_per_year]
    hold_prices = prices[4 * ticks_per_year:5 * ticks_per_year]

    for beta in betas:
        for k_frac in ks:
            for threshold in thresholds:
                strategy = MyStrategy()
                strategy.beta = beta
                strategy.k_frac = k_frac
                strategy.threshold = threshold

                result = run_backtest(train_prices, hold_prices, strategy, meta)
                dr = result["daily_returns"]
                eq = np.cumprod(1.0 + dr)
                dd = eq / np.maximum.accumulate(eq) - 1.0

                rows.append({
                    "beta": beta,
                    "k_frac": k_frac,
                    "threshold": threshold,
                    "sharpe": annualized_sharpe(dr),
                    "total_return": np.prod(1.0 + dr) - 1.0,
                    "max_drawdown": dd.min(),
                    "total_cost": np.sum(result["daily_costs"]),
                })

    return pd.DataFrame(rows).sort_values("sharpe", ascending=False)

In [ ]:
grid_results = evaluate_param_grid(
    prices,
    meta,
    betas=[0.10, 0.15, 0.20, 0.25],
    mom_lookbacks=[10, 20, 40],
    cov_lookbacks=[42, 63, 84],
)

grid_results.head(15)

In [ ]:
best = grid_results.iloc[0]
best

In [ ]:
best_strategy = MyStrategy()
best_strategy.beta = float(best["beta"])
best_strategy.lookback_mom = int(best["lookback_mom"])
best_strategy.lookback_cov = int(best["lookback_cov"])

best_result = run_backtest(train_prices, hold_prices, best_strategy, meta)
best_daily_returns = best_result["daily_returns"]

print("Best Sharpe:", annualized_sharpe(best_daily_returns))
print("Best Total Return:", np.prod(1.0 + best_daily_returns) - 1.0)
print("Best Total Cost:", np.sum(best_result["daily_costs"]))

In [ ]:
best_equity = np.cumprod(1.0 + best_daily_returns)

plt.figure(figsize=(10, 5))
plt.plot(best_equity)
plt.title("Best Strategy Equity Curve")
plt.xlabel("Day")
plt.ylabel("Portfolio Value")
plt.grid(True)
plt.show()